# 03. 상동성 누출 — random split vs 가족/슈퍼패밀리 split

**등급**: 🟢 Mac 로컬 (CPU) · **예상 시간**: 1~3분 · **네트워크**: 불필요

**질문**: 진화적으로 관련된 서열이 train/test에 섞이면 성능이 얼마나 부풀려지는가?

**설계 (합성 데이터)**
- 슈퍼패밀리 40개 × 가족 5개 × 서열 12개 = 2,400개 서열 (길이 100).
- 가족 조상은 슈퍼패밀리 조상에서 50% 위치가 바뀐 것(먼 상동), 가족 구성원은 가족 조상에서 5~30% 위치가 바뀐 것(가까운 상동)이다.
- 라벨(0/1)은 **슈퍼패밀리 단위**로 정해진다. 70%의 슈퍼패밀리는 **일반화 가능한 신호**(공유 모티프 유무)로 라벨이 정해지고, 나머지 30%는 모티프와 무관한 임의 라벨이다. 그래서 모티프를 배우면 새 슈퍼패밀리에서도 어느 정도 맞힐 수 있지만, 임의 라벨은 **기억**으로만 맞힐 수 있다.
- 가족 구성원은 모티프를 포함한 **전체 서열**에 변이가 들어가므로, 분기가 클수록 모티프도 훼손된다(신호 약화).
- 분류기는 3-mer 빈도 + 로지스틱 회귀(단순 baseline)다.

> 합성 데이터의 수치는 개념 시연용이다. 실제 단백질에서의 크기는 데이터·라벨·임계값에 따라 다르다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

V, L = 20, 100
N_SUPER, N_FAM, N_MEM = 40, 5, 12
rs = np.random.RandomState(0)

MOTIF = rs.randint(0, V, 8)  # 공유 모티프 (길이 8)


def mutate(seq, rate, rs):
    seq = seq.copy()
    mask = rs.rand(seq.size) < rate
    seq[mask] = rs.randint(0, V, mask.sum())   # 바뀐 위치는 무작위 잔기 (원래와 같을 수도 있어 실제 변화율 ≈ rate × 0.95)
    return seq


def insert_motif(seq, rs):
    pos = rs.randint(0, L - len(MOTIF))
    seq = seq.copy()
    seq[pos:pos + len(MOTIF)] = MOTIF
    return seq


X, y, fam, sup = [], [], [], []
for s in range(N_SUPER):
    anc = rs.randint(0, V, L)
    has_motif = rs.rand() < 0.5
    is_signal = rs.rand() < 0.7
    label = int(has_motif) if is_signal else int(rs.rand() < 0.5)
    for f in range(N_FAM):
        fam_anc = mutate(anc, 0.5, rs)
        if has_motif:
            fam_anc = insert_motif(fam_anc, rs)
        for _ in range(N_MEM):
            X.append(mutate(fam_anc, rs.uniform(0.05, 0.30), rs))
            y.append(label)
            fam.append(s * N_FAM + f)
            sup.append(s)
X, y, fam, sup = np.array(X), np.array(y), np.array(fam), np.array(sup)
print(X.shape, "positives:", y.mean().round(2), "| families:", len(set(fam)), "| superfamilies:", len(set(sup)))

## 1. 서열 동일도와 클러스터링
치환만 있으므로 같은 길이의 서열은 위치별 일치 비율(Hamming 동일도)로 비교할 수 있다. 원-핫 행렬 곱으로 한 번에 계산한다.
(실제 데이터에서는 정렬이 필요하므로 MMseqs2 같은 도구를 쓴다.)

In [ ]:
onehot = np.zeros((X.shape[0], L * V), dtype=np.float32)
onehot[np.arange(X.shape[0])[:, None], np.arange(L)[None, :] * V + X] = 1.0
IDENT = onehot @ onehot.T / L
np.fill_diagonal(IDENT, 0)


def same(a, b):
    return a[:, None] == b[None, :]


iu = np.triu_indices(len(X), 1)
pair_ident = IDENT[iu]
for name, mask in [("같은 가족", same(fam, fam)[iu]),
                   ("같은 슈퍼패밀리, 다른 가족", (same(sup, sup) & ~same(fam, fam))[iu]),
                   ("다른 슈퍼패밀리", ~same(sup, sup)[iu])]:
    q = np.percentile(pair_ident[mask], [5, 50, 95])
    print(f"{name:24s} 동일도 5/50/95%: {q[0]:.2f} / {q[1]:.2f} / {q[2]:.2f}")


def cluster_by_identity(thr):
    adj = sparse.csr_matrix(IDENT >= thr)
    n, labels = connected_components(adj, directed=False)
    return n, labels


for thr in [0.5, 0.3, 0.25, 0.2, 0.15]:
    n, _ = cluster_by_identity(thr)
    print(f"동일도 ≥ {thr:.2f} 단일 연결 클러스터 수: {n}")

**읽는 법 (seed=0 기준, 직접 확인할 것)**
- 같은 가족의 동일도는 중앙값 약 0.7, 같은 슈퍼패밀리의 **다른 가족**은 약 0.2, 다른 슈퍼패밀리는 약 0.05다.
- 임계값을 낮출수록 클러스터가 가족(200개) → 슈퍼패밀리(40개) 단위로 합쳐진다. 0.2 근처에서 클러스터 수가 슈퍼패밀리 수에 가까워진다.
- 0.15까지 낮추면 **단일 연결(single linkage)의 연쇄 효과**로 거의 모든 서열이 하나의 클러스터가 된다. 임계값이 너무 느슨하면 분할 자체가 불가능하다.
- 이것이 "split 임계값을 보고해야 하는" 이유다. 임계값이 느슨해야 먼 친척까지 분리되지만, 너무 느슨하면 분리할 대상이 사라진다.

## 2. 3-mer 특징과 split 3종
- **random**: 서열 단위 무작위 분할 (누출 가능)
- **family**: 가족 단위로 분할 (같은 가족은 같은 쪽, 먼 친척인 같은 슈퍼패밀리의 다른 가족은 train에 있을 수 있음)
- **cluster(0.20)**: 동일도 ≥ 0.20인 서열을 같은 클러스터로 묶고 클러스터 단위로 분할 (대략 슈퍼패밀리 수준까지 분리)
- **superfamily(true)**: 합성 데이터라 알고 있는 **정답 슈퍼패밀리**로 분할. 동일도 클러스터링이 이 정답에 얼마나 근접한지 비교하는 기준이다 (실제 데이터에는 이 정답이 없다).

In [ ]:
def kmer_counts(X, k=3):
    N, Ln = X.shape
    idx = sum(X[:, i:Ln - k + 1 + i] * (V ** (k - 1 - i)) for i in range(k))
    rows = np.repeat(np.arange(N), idx.shape[1])
    M = sparse.csr_matrix((np.ones(idx.size), (rows, idx.ravel())), shape=(N, V ** k))
    return M / idx.shape[1]


F = kmer_counts(X)
print("feature matrix:", F.shape)

n_cl, cluster020 = cluster_by_identity(0.20)
print("clusters at identity >= 0.20:", n_cl)
GROUPS = {"random": None, "family": fam, "cluster(0.20)": cluster020, "superfamily(true)": sup}


def split_indices(kind, seed, test_size=0.3):
    n = len(X)
    if kind == "random":
        r = np.random.RandomState(seed)
        perm = r.permutation(n)
        k = int(n * (1 - test_size))
        return perm[:k], perm[k:]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    return next(gss.split(F, y, GROUPS[kind]))


def run(kind, seed):
    tr, te = split_indices(kind, seed)
    if len(set(y[tr])) < 2 or len(set(y[te])) < 2:
        return None          # 한 클래스뿐이면 AUROC가 정의되지 않음 → 이 split은 버리고 다음 시드로 대체
    clf = LogisticRegression(max_iter=3000, C=50.0).fit(F[tr], y[tr])
    p = clf.predict_proba(F[te])[:, 1]
    max_id = IDENT[np.ix_(te, tr)].max(axis=1)         # 각 테스트 서열의 훈련 세트 최대 동일도
    return roc_auc_score(y[te], p), (p > 0.5) == y[te], max_id

## 3. 결과 — AUROC (시드 5회)

In [ ]:
N_REP = 5
res = {k: [] for k in GROUPS}
pooled = {k: ([], []) for k in GROUPS}       # (정답 여부, 최대 동일도) — 반복 평가를 이어붙인 것
skipped = {k: 0 for k in GROUPS}
for kind in GROUPS:
    sd, got = 0, 0
    while got < N_REP and sd < 100:
        out = run(kind, sd)
        sd += 1
        if out is None:
            skipped[kind] += 1
            continue
        auc, correct, max_id = out
        res[kind].append(auc)
        pooled[kind][0].append(correct)
        pooled[kind][1].append(max_id)
        got += 1
for kind, v in res.items():
    print(f"{kind:18s} AUROC {np.mean(v):.3f} ± {np.std(v):.3f}   (시드별: {np.round(v, 3)})   버려진 split: {skipped[kind]}")

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.bar(list(res), [np.mean(v) for v in res.values()], yerr=[np.std(v) for v in res.values()], capsize=4)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylabel("AUROC")
ax.set_ylim(0.4, 1.02)
plt.tight_layout()
plt.show()

**관찰 (시드 0~4, 직접 확인할 것)**
- random split의 AUROC가 가장 높고(약 0.98), 그룹 split에서는 낮아진다(약 0.87~0.91). 감소 폭은 합성 라벨 설계에 의존한다.
- 라벨의 70%는 공유 모티프로 정해지므로 그룹 split에서도 0.5보다 훨씬 높다. 나머지 30%(임의 라벨)는 같은 슈퍼패밀리의 친척이 훈련 세트에 있어야만 맞힐 수 있는 성분이다.
- **cluster(0.20)의 AUROC(약 0.87)는 정답 슈퍼패밀리 split(약 0.87)과 거의 같다.** 동일도 클러스터링이 이 합성 데이터에서는 정답 구조를 잘 복원했다는 뜻이다 (실제 데이터에는 정답이 없다).
- family와 cluster/superfamily의 AUROC 차이(약 0.03)는 시드별 표준편차(약 0.02~0.07)와 비슷한 크기라 **분할별 변동이 커서 차이의 불확실성이 크다.**
  시드 수를 늘리고 분기 정도·임의 라벨 비율을 바꿔 확인해야 한다 (과제 3).

## 4. 왜 그런가 — 훈련 세트와의 최대 동일도별 정확도
테스트 서열마다 훈련 세트의 가장 가까운 서열과의 동일도를 구하고, 그 구간별 정확도를 본다.

In [ ]:
bins = [0.0, 0.15, 0.25, 0.4, 0.6, 1.01]
labels = [f"{bins[i]:.2f}-{bins[i + 1]:.2f}" for i in range(len(bins) - 1)]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for kind in GROUPS:
    correct = np.concatenate(pooled[kind][0])
    mx = np.concatenate(pooled[kind][1])
    axes[0].hist(mx, bins=30, range=(0, 1), alpha=0.5, label=kind)
    accs, ns = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        sel = (mx >= lo) & (mx < hi)
        accs.append(correct[sel].mean() if sel.sum() >= 20 else np.nan)
        ns.append(int(sel.sum()))
    axes[1].plot(range(len(labels)), accs, marker="o", label=kind)
    print(f"{kind:18s} 구간별 n(5회 반복 평가 건수의 합, 고유 서열 수 아님): {dict(zip(labels, ns))}")
all_correct = np.concatenate([np.concatenate(pooled[k][0]) for k in GROUPS])
all_mx = np.concatenate([np.concatenate(pooled[k][1]) for k in GROUPS])
all_accs = []
for lo, hi in zip(bins[:-1], bins[1:]):
    sel = (all_mx >= lo) & (all_mx < hi)
    all_accs.append(all_correct[sel].mean() if sel.sum() >= 20 else np.nan)
axes[1].plot(range(len(labels)), all_accs, color="k", lw=2.5, marker="s", label="all splits pooled")
print("전체 합친 구간별 정확도:", dict(zip(labels, np.round(all_accs, 3))))
axes[0].set_xlabel("max identity to train")
axes[0].set_ylabel("# test sequences")
axes[0].legend()
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=30)
axes[1].set_ylabel("accuracy")
axes[1].set_xlabel("max identity to train (bins)")
axes[1].legend()
plt.tight_layout()
plt.show()

**읽는 법 (이 실행 기준, 직접 확인할 것)**
- 왼쪽: split마다 테스트 서열이 훈련 세트와 갖는 최대 동일도 분포가 다르다. random split은 높은 동일도(0.6 이상)에 몰려 있고, 그룹 split일수록 낮은 쪽으로 이동한다.
  그래서 **같은 동일도 구간끼리 split을 비교할 수는 없다**(구간이 거의 겹치지 않는다).
- 오른쪽 검정선: 모든 split의 반복 평가를 합쳐 동일도 구간별 정확도를 본 것이다. 동일도가 높을수록 정확도가 올라가는 패턴(이 실행에서 약 0.79 → 0.91)이 보인다.
  이는 "친척이 훈련 세트에 있을수록 맞히기 쉽다"는 설명과 **일치하는 패턴**이지, 메커니즘의 증명은 아니다(구간마다 포함된 가족·라벨 구성도 다르다).
- 구간 표본이 20개 미만이면 생략한다(NaN). n은 5회 반복 평가를 이어붙인 건수이며 고유 서열 수가 아니다.
- 보고된 성능이 어떤 동일도 영역에서 나온 것인지 모르면 숫자의 의미를 해석할 수 없다.

## 5. 실제 데이터에 적용 (코드 예시, 실행하지 않음)
실제 서열에서는 MMseqs2로 클러스터를 만들고 그 ID를 `groups`로 사용한다.

```bash
mmseqs easy-cluster seqs.fasta out tmp --min-seq-id 0.3 -c 0.8   # 옵션은 'mmseqs easy-cluster -h'로 확인
# out_cluster.tsv: (대표서열, 구성원) 두 열
```
주의할 점이 있다. 대표 서열 기반 클러스터링은 서로 다른 클러스터의 서열 쌍도 임계값 이상으로 유사할 수 있다. 엄격한 분리가 필요하면 train–test 쌍 단위로 최대 동일도를 직접 확인한다.
```python
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
cl = pd.read_csv("out_cluster.tsv", sep="\t", header=None, names=["rep", "member"])
group_of = dict(zip(cl.member, cl.rep))
groups = [group_of[name] for name in names]          # names: 서열 ID 목록
tr, te = next(GroupShuffleSplit(test_size=0.2, random_state=0).split(X, y, groups))
```

## 6. 합성 → 실제 전이 주의
- 이 실험에서 단순 3-mer 모델이 슈퍼패밀리 split에서도 AUROC ≈ 0.87을 낸 것은 **길이 8의 보존 모티프를 인위적으로 심었기 때문**이다. 연속 k-mer 신호가 먼 상동에서도 남도록 설계되어 있다.
- 실제 원거리 상동에서는 연속 k-mer가 잘 보존되지 않고 구조적·물리화학적 패턴만 남는 경우가 많아 k-mer baseline의 일반화가 훨씬 나빠질 수 있다(일반적으로 알려진 경향이며 이 노트북에서 검증하지 않았다).
- 따라서 이 노트북의 "감소 폭"을 실제 데이터의 예측치로 쓰면 안 된다. 여기서 배울 것은 **진단 절차**(동일도 분포 확인, 그룹 split, baseline 비교)다.

## 정리
1. random split 성능은 "아는 가족의 새 구성원"에 대한 성능이다. 평가 목표가 그것이라면 적절하지만, 새 가족에 대한 일반화를 말하려면 부풀려진 값이다.
2. 그룹을 어떻게 묶느냐(가족 vs 느슨한 클러스터)에 따라 결과가 달라진다. 임계값을 **보고**한다.
3. FM은 사전학습 데이터에서 이미 평가 서열의 친척을 봤을 수 있다. 이 노트북의 논리는 사전학습–평가 사이에도 그대로 적용된다.

**과제**: [../exercises.md](../exercises.md)의 과제 3.